# Joint multi-echo navigator/respiratory resolved structured low-rank (SLR) reconstruction updated for implementation into FIRE pipeline

Mark Chiew (mark.chiew@utoronto.ca)
The goal is to modify Mark's script so that the input is ismrmrd format and the output is a 16 bit image.

---
# Set up
---

In [1]:
# Imports
import numpy as np
from matplotlib import pyplot as plt

import sklearn   # used only for k-means
from sklearn import cluster
import gpuSLR    # GPU version of SLR
import SLR       # Structured low-rank methods for joint recon
import grappa    # grappa for computing initialization
import h5py      # added to handle ismrmrd data format
import ismrmrd   # added to handle ismrmrd data format
from common import SiemensRAW


# Define some helper functions

# handles centric k-space with shifting, along arbitrary dimensions
def fftdim(x, dims=None):
    return np.fft.fftshift(np.fft.fftn(np.fft.ifftshift(x), axes=dims, norm="ortho"))

def ifftdim(x, dims=None):
    return np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(x), axes=dims, norm="ortho"))

# root-sum-of-squares combination
def sos(x, axis=-1):
    return np.sqrt(np.sum(np.abs(x)**2, axis=axis))

def remove_oversampling(kspace, header):
    """
    Remove oversampling in k-space along a given dimension.
    """
    nx_encode = header.encoding[0].encodedSpace.matrixSize.x
    nx_readout = header.encoding[0].reconSpace.matrixSize.x

    if nx_encode > nx_readout:
        hybrid_space = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(kspace, axes=-2), axis=-2, norm="ortho"), axes=-2)
        start_idx = (nx_encode - nx_readout) // 2
        end_idx = start_idx + nx_readout
        
        crop_slices = [slice(None)] * kspace.ndim
        crop_slices[-2] = slice(start_idx, end_idx)
        hybrid_space_cropped = hybrid_space[tuple(crop_slices)]

        kspace_cropped = np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(hybrid_space_cropped, axes=-2), axis=-2, norm="ortho"), axes=-2)
        return kspace_cropped
    
    return kspace

---
# Load Data 
---
In the FIRE pipeline, data is emitted in ismrmrd format, without the reference scan.

In [19]:
file = 'output_again_2.h5'
dataset = ismrmrd.Dataset(file, 'dataset', False)

xml_header = dataset.read_xml_header()
mrd_header = ismrmrd.xsd.CreateFromDocument(xml_header)
raw_processor = SiemensRAW(mrd_header)

num_acquisitions = dataset.number_of_acquisitions()

for i in range(num_acquisitions):
    acq = dataset.read_acquisition(i)
    raw_processor.add_acq(acq)

In [ ]:
raw_processor.extract_noise()
raw_processor.remove_phase_stabilization_references()

kspace, navigator, acs_mask = raw_processor.build_kspace()
kspace = remove_oversampling(kspace, mrd_header)
navigator = remove_oversampling(navigator, mrd_header)


In [21]:
X = raw_processor.noise.data
W = np.linalg.cholesky(np.linalg.inv(np.cov(X)))
kspace = np.tensordot(kspace, W, axes=((-1,), (0,)))
img_nav = np.tensordot(navigator, W, axes=((-1,), (0,)))

In [ ]:
kspace = kspace.squeeze()
nrep, neco, nslc, ny, nx, nc = kspace.shape
R = int(mrd_header.encoding[0].parallelImaging.accelerationFactor.kspace_encoding_step_1)

(3, 4, 15, 384, 384, 4)


In [23]:
# slice to reconstruct
slc = 6

# echoes to reconstruct
eco = np.arange(neco)

# repetitions to reconstruct
rep = np.arange(nrep)

# number of bins to partition data into, based on navigator k-means clustering
# increasing this number increases the number of resolved dynamic states, but also increases computation time and memory
# on CPU, I would recommend nbins ≤ 8, anything beyond that gets pretty slow
# on GPU, I've tested up to nbins = 16 and it works reasonably fast, probably diminishing returns
nbins = 8

In [24]:
img_nav = img_nav.squeeze()
print(img_nav.shape)


(3, 15, 384, 384, 4)


In [28]:
# concatenate navigators, and inverse FFT navigators along RO dimension
# use only sampled lines, to avoid an extra trivial cluster from the empty lines
# need to check all the values here to see if i'm getting the right thing compared to mark 
nav = ifftdim(img_nav[:,slc,::R,:,:], dims=(-2,))
print(nav.shape)

# select RO indices near the spinal cord, as we only care about that region
sc_idx = np.arange(170,220)

# reference everything relative to the first line, and average the relative signals across coil channels
tmp = nav[:, :, sc_idx, :]
tmp = np.squeeze(np.mean(tmp*np.conj(tmp[:,[0],:,:]), axis=-1))

# because sk-learn k-means requires real-valued input, I've concatenated the real and imaginary parts of the navigator
tmp = np.concatenate((np.real(tmp), np.imag(tmp)), axis=-1)

# alternatively, you could try extracting just the phase of the navigator
#tmp = np.angle(tmp)

# get k-means cluster indices, with nbins clusters
idx = sklearn.cluster.KMeans(n_clusters=nbins).fit(tmp.reshape((-1,tmp.shape[-1]))).labels_.reshape((nrep,-1))
print(idx)

(3, 192, 384, 4)
[[4 5 7 3 3 3 4 5 1 4 3 3 7 5 5 7 3 3 3 4 5 1 3 3 3 4 5 7 3 3 3 4 1 7 3 3
  3 3 1 7 3 3 3 7 1 3 3 3 3 7 5 7 3 3 3 3 7 5 7 3 3 3 3 5 5 7 3 3 3 4 5 1
  4 3 3 3 1 1 4 3 3 7 5 7 3 3 3 3 1 1 3 3 5 3 3 1 4 3 7 4 3 5 3 3 3 1 1 4
  3 3 3 1 1 3 3 1 7 3 4 1 4 3 7 7 3 3 1 4 3 7 1 3 3 1 4 3 3 1 4 3 7 7 3 3
  7 7 3 3 1 4 3 4 1 3 3 7 4 3 4 1 3 3 7 7 3 4 1 4 3 7 7 3 4 7 3 3 7 1 4 3
  3 7 1 4 3 3 7 5 7 3 3 1]
 [4 3 0 0 4 5 1 4 0 0 0 7 5 7 3 0 0 0 4 1 4 0 0 0 3 1 1 3 0 0 0 3 1 1 3 0
  0 0 4 5 7 0 0 0 3 1 1 3 0 0 7 1 3 0 0 4 5 4 0 0 0 0 7 5 4 0 0 3 1 1 3 0
  0 0 1 1 3 0 0 0 1 1 4 0 0 0 0 7 1 4 0 0 1 0 4 4 0 4 7 0 0 5 0 0 0 0 7 5
  1 3 0 0 0 4 5 7 3 0 0 7 4 0 3 1 4 0 0 4 5 7 0 0 0 3 1 7 0 0 0 3 1 1 3 0
  0 3 1 1 3 0 0 3 1 1 3 0 0 0 0 7 5 7 0 0 0 4 5 1 3 0 0 0 7 5 7 0 0 0 0 1
  5 4 0 0 0 3 1 5 4 0 0 0]
 [4 1 4 6 2 2 2 6 7 1 0 6 2 2 6 7 4 6 2 2 0 7 3 2 2 2 3 7 6 2 2 0 7 0 2 2
  0 7 0 2 2 6 4 6 2 2 6 7 0 2 2 2 3 4 6 2 3 6 2 2 3 0 2 2 2 0 4 6 2 2 2 4
  3 2 2 2 2 4 3 6 2 2 0 7 6 2 2 6 4 6 2 6